In [1]:
!pip install vina meeko rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 43.9 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install gemmi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 69.3 MB/s eta 0:00:00:00:01


In [3]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.1 MB/s eta 0:00:00:00:01


In [4]:
!pip install pdbfixer openmm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 24.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 79.3 MB/s eta 0:00:00:00:0100:01


In [5]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

from rdkit import Chem
from meeko import MoleculePreparation  #Convert ligands to PDBQT format
from meeko import PDBQTWriterLegacy

from vina import Vina

from Bio.PDB import PDBParser #Used for reading protein structures

from pdbfixer import PDBFixer #Fix protein structures
from openmm.app import PDBFile #Write repaired PDB files

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
CORE_DIR = Path("/kaggle/input/datasets/madukacharles/pdbbind-protein-ligand-binding-affinity-dataset/v2013-core")

RESULT_DIR = Path("/kaggle/working/docking_results")

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True)

In [7]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

def fix_pocket(input_pdb, output_pdb):

    fixer = PDBFixer(filename=str(input_pdb))

    fixer.addMissingHydrogens(pH=7.4) #Crystal structures usually don't contain hydrogen atoms at pH 7.4, partially protonated

    with open(output_pdb, "w") as f:
        PDBFile.writeFile(fixer.topology,fixer.positions,f) 

In [8]:
def prepare_ligand(ligand_file):

    mol = Chem.SDMolSupplier(str(ligand_file),removeHs=False)[0] #SDF reader

    prep = MoleculePreparation() #Meeko preparation object

    setup = prep.prepare(mol)[0] # understands the whole mol

    ligand_pdbqt, success, error_msg = (PDBQTWriterLegacy.write_string(setup)) # convert pdb -> pdbqt 

    return ligand_pdbqt

In [9]:
def prepare_receptor(pocket_pdb,receptor_prefix):

    subprocess.run(
        [
            "mk_prepare_receptor.py",  #prepares receptors for docking
            "--read_pdb",
            str(pocket_pdb),
            "-o",  # output
            str(receptor_prefix),
            "-p" #pdbqt
        ],check=True)

    return f"{receptor_prefix}.pdbqt"

In [10]:
def compute_box(pocket_file):

    pocket = PDBParser(QUIET=True).get_structure("pocket",str(pocket_file)) 

    coords = np.array([
        atom.get_coord()
        for atom in pocket.get_atoms()])   # gets cords for each atom

    center = coords.mean(axis=0)  # cal the mean of all atoms

    size = (coords.max(axis=0) - coords.min(axis=0)) + 5 # box size with paddig 

    return center, size

Load files
Prepare ligand
Prepare receptor
Create docking box
Run Vina
Save best pose
Return affinity

In [11]:
def dock_one_complex(pdb_id):

    complex_dir = CORE_DIR / pdb_id

    ligand_file = (complex_dir / f"{pdb_id}_ligand.sdf")

    pocket_file = (complex_dir / f"{pdb_id}_pocket.pdb")

    output_dir = RESULT_DIR / pdb_id

    output_dir.mkdir(exist_ok=True)


    ligand_pdbqt = prepare_ligand(ligand_file)

    fixed_pocket = (output_dir /"fixed_pocket.pdb")

    fixer = PDBFixer(filename=str(pocket_file))
    fixer.addMissingHydrogens( pH=7.4 ) #Adds all missing H atom

    with open(fixed_pocket, "w") as f:

        PDBFile.writeFile(fixer.topology,fixer.positions,f)

    receptor_prefix = (output_dir / "receptor" )

    receptor_pdbqt = prepare_receptor(fixed_pocket,receptor_prefix)

    center, size = compute_box(fixed_pocket)

    v = Vina(sf_name="vina") #Creates docking engine

    v.set_receptor(receptor_pdbqt)  # Load receptor structure

    v.set_ligand_from_string(ligand_pdbqt)   # Load ligand structure

    v.compute_vina_maps(center=center.tolist(),box_size=size.tolist()) #Creates docking engine

    v.dock(exhaustiveness=8, n_poses=9) #Perform Docking

    pose_file = ( output_dir / "best_pose.pdbqt")

    v.write_pose( str(pose_file),overwrite=True)

    energies = v.energies()

    best_affinity = float(energies[0][0])

    return {
        "pdb_id": pdb_id,
        "affinity": best_affinity,
        "pose_file": str(pose_file)
    }